# Explore Fixations

**Purpose:** Exploratory analysis comparing fixation durations between two conditions (e.g. stair vs hill).

This notebook is *not* part of the main pipeline — it is a standalone exploration tool for investigating fixation behaviour from Pupil Neon exports.

## What it does

1. Load two `fixations.csv` files (one per condition).
2. Filter out unrealistically long fixations (> 1250 ms).
3. Compare fixation-duration distributions side by side.
4. Compute fixation rates (fixations per second).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import datetime
import pandas as pd

from neon_gaze.io import load_fixations
from neon_gaze.plotting import plot_fixation_histograms

## Configuration

Set `DEMO = True` if you want to test with placeholder paths (no demo fixation data is included).  
Set `DEMO = False` (the default) and update the paths below to point at your own `fixations.csv` exports.


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# If you are using this repo for the first time, set DEMO = True.
# Note: no demo fixation data is included — you must supply your own.
DEMO = False

if DEMO:
    STAIR_FIXATIONS_PATH = "../demo/input/fixations_stair.csv"  # not included
    HILL_FIXATIONS_PATH  = "../demo/input/fixations_hill.csv"   # not included
else:
    STAIR_FIXATIONS_PATH = "../data/session_01/fixations.csv"
    HILL_FIXATIONS_PATH  = "../data/session_02/fixations.csv"

# Maximum fixation duration to keep (ms)
MAX_DURATION_MS = 1250


## Load and filter fixations

In [ ]:
stair_df = load_fixations(STAIR_FIXATIONS_PATH)
hill_df = load_fixations(HILL_FIXATIONS_PATH)

# Filter out fixations exceeding the duration threshold
stair_df = stair_df[stair_df["duration [ms]"] <= MAX_DURATION_MS].reset_index(drop=True)
hill_df = hill_df[hill_df["duration [ms]"] <= MAX_DURATION_MS].reset_index(drop=True)

print(f"Stair fixations kept: {len(stair_df)}")
print(f"Hill fixations kept:  {len(hill_df)}")

## Side-by-side duration histograms

In [ ]:
plot_fixation_histograms(
    df_left=stair_df,
    df_right=hill_df,
    labels=("Stair", "Hill"),
    bins=50,
    title_string="Fixation Durations",
)

## Collection duration and fixation rate

In [ ]:
def compute_collection_duration(df, start_col="start timestamp [ns]", end_col="end timestamp [ns]"):
    """Return (duration_seconds, info_dict) from earliest start to latest end."""
    start_vals = pd.to_numeric(df[start_col], errors="coerce")
    end_vals = pd.to_numeric(df[end_col], errors="coerce")
    first_start = start_vals.min()
    last_end = end_vals.max()
    if pd.isna(first_start) or pd.isna(last_end):
        return None
    duration_ns = int(last_end - first_start)
    duration_s = duration_ns / 1e9
    return duration_s, {
        "first_start_ns": int(first_start),
        "last_end_ns": int(last_end),
        "duration_s": duration_s,
        "duration_hms": str(datetime.timedelta(seconds=duration_s)),
    }


stair_dur = compute_collection_duration(stair_df)
hill_dur = compute_collection_duration(hill_df)

for label, dur, n in [("Stair", stair_dur, len(stair_df)), ("Hill", hill_dur, len(hill_df))]:
    if dur is not None:
        rate = n / dur[0] if dur[0] > 0 else None
        print(f"{label}: {n} fixations, duration = {dur[1]['duration_hms']}, rate = {rate:.3f} fix/sec")
    else:
        print(f"{label}: could not compute duration")